In [1]:
import pandas as pd

acc = pd.read_csv('accepted_with_reviews.csv')
rej = pd.read_csv('rejected_with_reviews.csv')
rc  = pd.read_csv('pr_review_comments_677.csv')

print(f'accepted : {len(acc):,} rows')
print(f'rejected : {len(rej):,} rows')
print(f'comments : {len(rc):,} rows')

accepted : 842 rows
rejected : 95 rows
comments : 4,470 rows


In [2]:
print('=== accepted ===')
print(f'Shape     : {acc.shape}')
print(f'Columns   : {acc.columns.tolist()}')
print(f'Agents    : {acc["agent"].value_counts().to_dict()}')
print(f'Nulls     :\n{acc[["id","title","body","agent","user","html_url"]].isna().sum().to_string()}')

print('\n=== rejected ===')
print(f'Shape     : {rej.shape}')
print(f'Columns   : {rej.columns.tolist()}')
print(f'Agents    : {rej["agent"].value_counts().to_dict()}')
print(f'Nulls     :\n{rej[["id","title","body","agent","user","html_url"]].isna().sum().to_string()}')

print('\n=== comments ===')
print(f'Shape         : {rc.shape}')
print(f'Columns       : {rc.columns.tolist()}')
print(f'Unique pr_ids : {rc["pr_id"].nunique()}')
print(f'user_type     : {rc["user_type"].value_counts().to_dict()}')
print(f'Nulls         :\n{rc[["id","pr_id","user","user_type","body"]].isna().sum().to_string()}')

print('\n=== linkage ===')
acc_with = acc['id'].isin(rc['pr_id']).sum()
rej_with = rej['id'].isin(rc['pr_id']).sum()
print(f'Accepted with comments    : {acc_with} / {len(acc)} ({acc_with/len(acc)*100:.1f}%)')
print(f'Accepted without comments : {len(acc)-acc_with} / {len(acc)} ({(len(acc)-acc_with)/len(acc)*100:.1f}%)')
print(f'Rejected with comments    : {rej_with} / {len(rej)} ({rej_with/len(rej)*100:.1f}%)')
print(f'Rejected without comments : {len(rej)-rej_with} / {len(rej)} ({(len(rej)-rej_with)/len(rej)*100:.1f}%)')

=== accepted ===
Shape     : (842, 22)
Columns   : ['id', 'number', 'title', 'body', 'agent', 'user_id', 'user', 'state', 'created_at', 'closed_at', 'merged_at', 'repo_id', 'repo_url', 'html_url', 'node_id', 'user_login', 'user_type', 'updated_at', 'is_merged', 'repo_full_name', 'labels', 'collected_at']
Agents    : {'Copilot': 268, 'Devin': 242, 'Claude_Code': 178, 'OpenAI_Codex': 105, 'Cursor': 49}
Nulls     :
id            0
title         0
body          5
agent         0
user        457
html_url      0

=== rejected ===
Shape     : (95, 22)
Columns   : ['id', 'number', 'title', 'body', 'agent', 'user_id', 'user', 'state', 'created_at', 'closed_at', 'merged_at', 'repo_id', 'repo_url', 'html_url', 'node_id', 'user_login', 'user_type', 'updated_at', 'is_merged', 'repo_full_name', 'labels', 'collected_at']
Agents    : {'Devin': 46, 'Claude_Code': 19, 'Copilot': 19, 'OpenAI_Codex': 8, 'Cursor': 3}
Nulls     :
id           0
title        0
body         0
agent        0
user        34
htm

In [5]:
# Group human comments by pr_id for fast lookup
human_comments_by_pr = human_rc.groupby('pr_id')

def get_reviewer_comments(pr_id, pr_author):
    """Return list of (user, body) — human only, excluding PR author."""
    if pr_id not in human_comments_by_pr.groups:
        return []
    group = human_comments_by_pr.get_group(pr_id)
    comments = []
    for _, row in group.iterrows():
        user = str(row['user']).strip()
        body = str(row['body']).strip()
        if body and user.lower() != str(pr_author).lower():
            comments.append((user, body))
    return comments

# Filter to PRs with at least 1 human comment
acc_study = acc[acc['id'].isin(human_rc['pr_id'])].copy().reset_index(drop=True)
rej_study = rej[rej['id'].isin(human_rc['pr_id'])].copy().reset_index(drop=True)

print(f'Accepted PRs with human reviews : {len(acc_study)} / {len(acc)}')
print(f'Rejected PRs with human reviews : {len(rej_study)} / {len(rej)}')
print(f'\nThese are the PRs we will classify.')
print(f'PRs with zero human comments are excluded from the study.')

Accepted PRs with human reviews : 245 / 842
Rejected PRs with human reviews : 44 / 95

These are the PRs we will classify.
PRs with zero human comments are excluded from the study.


In [ ]:
importinggg the anthropic


In [11]:
SYSTEM_PROMPT = """
You are a software engineering researcher classifying pull request reviewer comments
using the AlOmar et al. (2022) Refactoring Review Taxonomy.

TAXONOMY (use ONLY these — never invent new categories):

QUALITY:
  - Code smell
  - Internal quality attribute
  - External quality attribute
  - Technical debt
  - Design pattern
  - Coding convention
  - Lack of documentation

REFACTORING:
  - Refactoring correctness
  - Behavior preservation
  - Refactoring co-occurrence with changes
  - Domain constraint

OBJECTIVE:
  - Unclear goal
  - Unknown benefit
  - Potential side effect
  - Scope change
  - Feature support
  - Bug fixing

TESTING:
  - Lack of coverage
  - Absence of result
  - Poor test quality

INTEGRATION:
  - Configuration issue
  - Merge conflict
  - API management
  - Build failure

MANAGEMENT:
  - No review activity
  - Forgotten review
  - Change dependency
  - Review guideline violation

RULES:
1. Classify based on REVIEWER comments only — not the PR description or author notes.
2. If all reviewers gave only simple approvals (LGTM, thumbs up, emoji, short praise) with NO substantive critique → MANAGEMENT → No review activity
3. If there are NO human reviewer comments → MANAGEMENT → No review activity
4. Multiple taxonomy categories allowed if PR has multiple distinct reviewer concerns.
5. Evidence must be a direct quote from a reviewer comment.

OUTPUT: respond ONLY with valid JSON, no extra text:
{
  "classifications": [
    {
      "main_category": "MANAGEMENT",
      "subcategory": "No review activity",
      "evidence": "LGTM! 👍",
      "explanation": "All reviewers gave simple approvals with no substantive critique."
    }
  ]
}
""".strip()

def classify_pr(pr_title, pr_url, reviewer_comments):
    if not reviewer_comments:
        return [{
            "main_category": "MANAGEMENT",
            "subcategory": "No review activity",
            "evidence": "No human reviewer comments found.",
            "explanation": "No human reviewers left any comments on this PR."
        }]

    formatted = "\n".join(f"@{user}: {body[:300]}" for user, body in reviewer_comments)

    user_msg = f"""PR Title: {pr_title}
PR URL: {pr_url}

REVIEWER COMMENTS:
{formatted}

Classify the reviewer feedback using the AlOmar Refactoring Review Taxonomy."""

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=500,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_msg}]
    )

    raw = response.content[0].text.strip()
    raw = re.sub(r'^```json\s*', '', raw)
    raw = re.sub(r'```$', '', raw).strip()

    data = json.loads(raw)
    return data.get("classifications", [])

print("Classification function ready.")

Classification function ready.


In [ ]:
code of loading the api

PR     : https://github.com/sog4be/sakurs/pull/76
Title  : test: enhance sakurs-core test coverage with comprehensive unit and integration tests
Comments found: 0

--- Classifying... ---

  Main     : MANAGEMENT
  Sub      : No review activity
  Evidence : No human reviewer comments found.


In [14]:
# Group human comments by pr_id
human_comments_by_pr = human_rc.groupby('pr_id')

def get_reviewer_comments(pr_id, pr_author):
    """Return list of (user, body) — human only, excluding PR author."""
    if pr_id not in human_comments_by_pr.groups:
        return []
    group = human_comments_by_pr.get_group(pr_id)
    comments = []
    for _, row in group.iterrows():
        user = str(row['user']).strip()
        body = str(row['body']).strip()
        if body and user.lower() != str(pr_author).lower():
            comments.append((user, body))
    return comments

# Verify on the test PR
test_row = acc_study.iloc[0]
comments = get_reviewer_comments(test_row['id'], test_row['user'])

print(f'PR     : {test_row["html_url"]}')
print(f'Author : {test_row["user"]}')
print(f'All human comments   : {len(human_rc[human_rc["pr_id"] == test_row["id"]])}')
print(f'From author only     : {len(human_rc[(human_rc["pr_id"] == test_row["id"]) & (human_rc["user"] == test_row["user"])])}')
print(f'Independent reviewer comments : {len(comments)}')
print()

# Now check how many PRs actually have independent reviewer comments
acc_independent = []
for _, row in acc_study.iterrows():
    c = get_reviewer_comments(row['id'], row['user'])
    if len(c) > 0:
        acc_independent.append(row['id'])

rej_independent = []
for _, row in rej_study.iterrows():
    c = get_reviewer_comments(row['id'], row['user'])
    if len(c) > 0:
        rej_independent.append(row['id'])

print(f'Accepted PRs with independent reviewer comments : {len(acc_independent)} / {len(acc_study)}')
print(f'Rejected PRs with independent reviewer comments : {len(rej_independent)} / {len(rej_study)}')
print(f'\nThe rest → MANAGEMENT / No review activity')

PR     : https://github.com/sog4be/sakurs/pull/76
Author : sog4be
All human comments   : 11
From author only     : 11
Independent reviewer comments : 0

Accepted PRs with independent reviewer comments : 221 / 245
Rejected PRs with independent reviewer comments : 44 / 44

The rest → MANAGEMENT / No review activity


In [15]:
# Update study dataframes to all PRs (those with and without independent comments)
# PRs without independent reviewer comments will be auto-tagged MANAGEMENT / No review activity

acc_with_independent = acc[acc['id'].isin(acc_independent)].copy().reset_index(drop=True)
rej_with_independent = rej[rej['id'].isin(rej_independent)].copy().reset_index(drop=True)

print(f'PRs to classify (have independent reviewer comments):')
print(f'  Accepted : {len(acc_with_independent)}')
print(f'  Rejected : {len(rej_with_independent)}')

print(f'\nPRs auto-tagged as No review activity:')
print(f'  Accepted : {len(acc) - len(acc_with_independent)}')
print(f'  Rejected : {len(rej) - len(rej_with_independent)}')

print(f'\nTotal PRs in study:')
print(f'  Accepted : {len(acc)}')
print(f'  Rejected : {len(rej)}')

PRs to classify (have independent reviewer comments):
  Accepted : 221
  Rejected : 44

PRs auto-tagged as No review activity:
  Accepted : 621
  Rejected : 51

Total PRs in study:
  Accepted : 842
  Rejected : 95


In [19]:
test_row = acc_with_independent.iloc[0]
test_comments = get_reviewer_comments(test_row['id'], test_row['user'])

print(f'PR     : {test_row["html_url"]}')
print(f'Title  : {test_row["title"]}')
print(f'Author : {test_row["user"]}')
print(f'Comments found: {len(test_comments)}')
for user, body in test_comments[:3]:
    print(f'  @{user}: {body[:150]}')

PR     : https://github.com/replicatedhq/platform-examples/pull/79
Title  : fix: refactor PR Validation workflow to use Replicated actions
Author : adamancini
Comments found: 6
  @banjoh: +1 on using cache
  @banjoh: Its not clear why we are marking this action as deprecated instead of removing it. Is it relied on elsewhere, hence needing to keep it?

Wouldn't it
  @banjoh: ```suggestion
    runs-on: latest
```

It can also be `ubuntu-24.04` if we want to track specific versions


In [20]:
result = classify_pr_with_retry(test_row['title'], test_row['html_url'], test_comments)

for r in result:
    print(f'Main     : {r["main_category"]}')
    print(f'Sub      : {r["subcategory"]}')
    print(f'Evidence : {r["evidence"][:120]}')
    print()

Main     : OBJECTIVE
Sub      : Unclear goal
Evidence : Its not clear why we are marking this action as deprecated instead of removing it. Is it relied on elsewhere, hence need

Main     : QUALITY
Sub      : Coding convention
Evidence : runs-on: latest - It can also be `ubuntu-24.04` if we want to track specific versions

Main     : TESTING
Sub      : Lack of coverage
Evidence : We have some customers on `1.29`, some older. Would it be better to cover these configurations?

Main     : OBJECTIVE
Sub      : Feature support
Evidence : Was the [create cluster](https://github.com/replicatedhq/replicated-actions/tree/main/create-cluster) action not fit for



In [24]:
def classify_pr(pr_title, pr_url, reviewer_comments):
    if not reviewer_comments:
        return [{
            'main_category': 'MANAGEMENT',
            'subcategory':   'No review activity',
            'evidence':      'No independent reviewer comments found.',
            'explanation':   'All comments were from the PR author or no human comments exist.'
        }]

    # Cap at 10 comments, 150 chars each to avoid token overflow
    capped_comments = reviewer_comments[:10]
    formatted = '\n'.join(f'@{user}: {body[:150]}' for user, body in capped_comments)

    user_msg = f"""PR Title: {pr_title}
PR URL: {pr_url}

REVIEWER COMMENTS:
{formatted}

Classify the reviewer feedback using the AlOmar Refactoring Review Taxonomy."""

    response = client.messages.create(
        model='claude-sonnet-4-5',
        max_tokens=500,
        system=SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': user_msg}]
    )

    raw = response.content[0].text.strip()
    raw = re.sub(r'^```json\s*', '', raw)
    raw = re.sub(r'```$',        '', raw).strip()

    data = json.loads(raw)
    return data.get('classifications', [])

In [25]:
test_row_rej = rej_with_independent.iloc[0]
test_comments_rej = get_reviewer_comments(test_row_rej['id'], test_row_rej['user'])

print(f'PR     : {test_row_rej["html_url"]}')
print(f'Title  : {test_row_rej["title"]}')
print(f'Author : {test_row_rej["user"]}')
print(f'Comments found: {len(test_comments_rej)}')
for user, body in test_comments_rej[:3]:
    print(f'  @{user}: {body[:150]}')

print('\n--- Classifying... ---')
result_rej = classify_pr_with_retry(test_row_rej['title'], test_row_rej['html_url'], test_comments_rej)
for r in result_rej:
    print(f'Main     : {r["main_category"]}')
    print(f'Sub      : {r["subcategory"]}')
    print(f'Evidence : {r["evidence"][:120]}')
    print()

PR     : https://github.com/OpenCut-app/OpenCut/pull/479
Title  : Try to help but need some help
Author : donghaozhang
Comments found: 24
  @timothyr: We can remove this script since the editor is fixed
  @timothyr: Why skip auth for desktop?
  @TenzDelek: remove the log

--- Classifying... ---
Main     : QUALITY
Sub      : Coding convention
Evidence : use es6 man, still stuck in require

Main     : QUALITY
Sub      : Code smell
Evidence : remove the log

Main     : OBJECTIVE
Sub      : Unclear goal
Evidence : Why skip auth for desktop?

Main     : OBJECTIVE
Sub      : Scope change
Evidence : We can remove this script since the editor is fixed

Main     : QUALITY
Sub      : Design pattern
Evidence : i prefer react icon



In [35]:
import pandas as pd

old_acc = pd.read_csv('taxonomy_accepted.csv')
old_rej = pd.read_csv('taxonomy_rejected.csv')

# Find new PRs that were NOT in the old run
new_acc_to_classify = acc_with_independent[~acc_with_independent['html_url'].isin(old_acc['PR_Link'])].copy().reset_index(drop=True)
new_rej_to_classify = rej_with_independent[~rej_with_independent['html_url'].isin(old_rej['PR_Link'])].copy().reset_index(drop=True)

print(f'Old accepted already classified : {len(old_acc)}')
print(f'Old rejected already classified : {len(old_rej)}')
print()
print(f'New accepted to classify : {len(new_acc_to_classify)}')
print(f'New rejected to classify : {len(new_rej_to_classify)}')

Old accepted already classified : 275
Old rejected already classified : 47

New accepted to classify : 210
New rejected to classify : 32


In [36]:
# Seed checkpoints with old results so the loop skips already classified PRs
old_acc.to_csv(CHECKPOINT_ACC, index=False)
old_rej.to_csv(CHECKPOINT_REJ, index=False)

print(f'Checkpoint seeded:')
print(f'  {CHECKPOINT_ACC} → {len(old_acc)} PRs')
print(f'  {CHECKPOINT_REJ} → {len(old_rej)} PRs')

Checkpoint seeded:
  checkpoint_accepted.csv → 275 PRs
  checkpoint_rejected.csv → 47 PRs


In [37]:
print('Running accepted PRs...')
acc_results = run_classification(new_acc_to_classify, acc, 'accepted', CHECKPOINT_ACC)

print('\nRunning rejected PRs...')
rej_results = run_classification(new_rej_to_classify, rej, 'rejected', CHECKPOINT_REJ)

Running accepted PRs...
Resuming — 275 already done
  [276/210] platform-examples/pull/79 → OBJECTIVE ; QUALITY ; TESTING ; OBJECTIVE
  [277/210] pyretailscience/pull/122 → OBJECTIVE ; OBJECTIVE
  [278/210] memstat/pull/4 → OBJECTIVE ; OBJECTIVE
  [279/210] blackjax/pull/13 → QUALITY ; QUALITY ; OBJECTIVE ; REFACTORING
  [280/210] crun/pull/1781 → REFACTORING ; OBJECTIVE
  [281/210] interface/pull/163 → QUALITY
  [282/210] billy-bot/pull/19 → QUALITY ; REFACTORING
  [283/210] billy-bot/pull/23 → QUALITY


KeyboardInterrupt: 

In [40]:
print(f'  [{len(done)+len(results)}/{len(done)+total}] {pr_short} → {main_cats}')

NameError: name 'done' is not defined

In [42]:
def run_new_only(df_to_classify, label, checkpoint_path):
    try:
        done = pd.read_csv(checkpoint_path)
        done_ids = set(done['PR_Link'])
        print(f'Resuming — {len(done)} already done')
    except FileNotFoundError:
        done = pd.DataFrame()
        done_ids = set()

    results = []
    total = len(df_to_classify)

    for i, row in df_to_classify.iterrows():
        if row['html_url'] in done_ids:
            continue

        comments = get_reviewer_comments(row['id'], row['user'])
        classifications = classify_pr_with_retry(row['title'], row['html_url'], comments)

        main_cats    = ' ; '.join(c['main_category'] for c in classifications)
        subcats      = ' ; '.join(c['subcategory']   for c in classifications)
        evidences    = ' | '.join(c.get('evidence',    '')[:150] for c in classifications)
        explanations = ' | '.join(c.get('explanation', '')[:200] for c in classifications)

        pr_short = '/'.join(row['html_url'].split('/')[-3:])

        results.append({
            'PR':            pr_short,
            'PR_Link':       row['html_url'],
            'Main_Category': main_cats,
            'Subcategory':   subcats,
            'Evidence':      evidences,
            'Explanation':   explanations,
            'Label':         label,
            'Status':        'Accepted' if label == 'accepted' else 'Rejected'
        })

        print(f'  [{len(done)+len(results)}/{len(done)+total}] {pr_short} → {main_cats}')

        if results and len(results) % 20 == 0:
            checkpoint = pd.concat([done, pd.DataFrame(results)], ignore_index=True)
            checkpoint.to_csv(checkpoint_path, index=False)
            print(f'  ✅ checkpoint saved at {len(done)+len(results)} PRs')

        time.sleep(0.3)

    final = pd.concat([done, pd.DataFrame(results)], ignore_index=True)
    final['#'] = range(1, len(final) + 1)
    final.to_csv(checkpoint_path, index=False)
    print(f'\n✅ [{label}] Done — {len(final)} total classified')
    return final


print('Running NEW accepted PRs...')
acc_new_results = run_new_only(new_acc_to_classify, 'accepted', CHECKPOINT_ACC)

print('\nRunning NEW rejected PRs...')
rej_new_results = run_new_only(new_rej_to_classify, 'rejected', CHECKPOINT_REJ)

Running NEW accepted PRs...
Resuming — 275 already done
  [276/485] platform-examples/pull/79 → OBJECTIVE ; QUALITY ; TESTING ; OBJECTIVE
  [277/485] pyretailscience/pull/122 → OBJECTIVE ; QUALITY
  [278/485] memstat/pull/4 → OBJECTIVE ; OBJECTIVE
  [279/485] blackjax/pull/13 → QUALITY ; QUALITY ; QUALITY ; OBJECTIVE
  [280/485] crun/pull/1781 → QUALITY ; OBJECTIVE
  [281/485] interface/pull/163 → QUALITY
  [282/485] billy-bot/pull/19 → QUALITY
  [283/485] billy-bot/pull/23 → QUALITY
  [284/485] variegated-rs/pull/7 → QUALITY ; QUALITY
  [285/485] bibo-note/pull/30 → QUALITY ; QUALITY ; OBJECTIVE
  [286/485] github-materials/pull/20 → QUALITY
  [287/485] winforms/pull/4 → QUALITY
  [288/485] tabnet/pull/115 → REFACTORING ; OBJECTIVE
  [289/485] Sophia/pull/31 → QUALITY ; OBJECTIVE
  [290/485] bahk/pull/188 → QUALITY ; OBJECTIVE
  [291/485] ai/pull/307 → QUALITY ; OBJECTIVE
  [292/485] capture-app/pull/4181 → REFACTORING ; OBJECTIVE ; REFACTORING ; OBJECTIVE
  Attempt 1 failed: Expectin

In [43]:
# Find all ERROR rows
err_acc = acc_new_results[acc_new_results['Main_Category'] == 'ERROR'].copy()
err_rej = rej_new_results[rej_new_results['Main_Category'] == 'ERROR'].copy()

print(f'Accepted ERRORs : {len(err_acc)}')
print(f'Rejected ERRORs : {len(err_rej)}')
print()
print(err_acc[['PR', 'PR_Link']].to_string())
print()
print(err_rej[['PR', 'PR_Link']].to_string())


Accepted ERRORs : 22
Rejected ERRORs : 5

                                  PR                                                      PR_Link
292            capture-app/pull/4109               https://github.com/dhis2/capture-app/pull/4109
300            capture-app/pull/4112               https://github.com/dhis2/capture-app/pull/4112
311            capture-app/pull/4244               https://github.com/dhis2/capture-app/pull/4244
320                    flux/pull/204                 https://github.com/harnesslabs/flux/pull/204
324                 language/pull/13      https://github.com/bold-through-things/language/pull/13
354      mcp-config-converter/pull/6        https://github.com/jr2804/mcp-config-converter/pull/6
389            neuro-san-ui/pull/107    https://github.com/cognizant-ai-lab/neuro-san-ui/pull/107
393               airbyte/pull/72395              https://github.com/airbytehq/airbyte/pull/72395
398        keboola-as-code/pull/2529         https://github.com/keboola/kebo

In [44]:
def classify_pr_high_tokens(pr_title, pr_url, reviewer_comments, retries=3):
    """Same as classify_pr but with higher max_tokens for complex PRs."""
    if not reviewer_comments:
        return [{
            'main_category': 'MANAGEMENT',
            'subcategory':   'No review activity',
            'evidence':      'No independent reviewer comments found.',
            'explanation':   'All comments were from the PR author or no human comments exist.'
        }]

    capped_comments = reviewer_comments[:10]
    formatted = '\n'.join(f'@{user}: {body[:100]}' for user, body in capped_comments)

    user_msg = f"""PR Title: {pr_title}
PR URL: {pr_url}

REVIEWER COMMENTS:
{formatted}

Classify the reviewer feedback using the AlOmar Refactoring Review Taxonomy."""

    for attempt in range(retries):
        try:
            response = client.messages.create(
                model='claude-sonnet-4-5',
                max_tokens=2048,
                system=SYSTEM_PROMPT,
                messages=[{'role': 'user', 'content': user_msg}]
            )
            raw = response.content[0].text.strip()
            raw = re.sub(r'^```json\s*', '', raw)
            raw = re.sub(r'```$', '', raw).strip()
            data = json.loads(raw)
            return data.get('classifications', [])
        except Exception as e:
            print(f'  Attempt {attempt+1} failed: {e}')
            if attempt < retries - 1:
                time.sleep(2)

    return [{
        'main_category': 'ERROR',
        'subcategory':   'ERROR',
        'evidence':      'Classification failed after retries.',
        'explanation':   'API error after 3 attempts.'
    }]


# Fix ERROR rows in accepted
all_errors = pd.concat([
    err_acc[['PR_Link']].assign(dataset='acc'),
    err_rej[['PR_Link']].assign(dataset='rej')
], ignore_index=True)

fixed_acc = []
fixed_rej = []

for _, err_row in all_errors.iterrows():
    url = err_row['PR_Link']
    dataset = err_row['dataset']

    # Find original PR data
    pr_row = acc[acc['html_url'] == url] if dataset == 'acc' else rej[rej['html_url'] == url]
    if pr_row.empty:
        continue
    pr_row = pr_row.iloc[0]

    comments = get_reviewer_comments(pr_row['id'], pr_row['user'])
    classifications = classify_pr_high_tokens(pr_row['title'], pr_row['html_url'], comments)

    main_cats    = ' ; '.join(c['main_category'] for c in classifications)
    subcats      = ' ; '.join(c['subcategory']   for c in classifications)
    evidences    = ' | '.join(c.get('evidence',    '')[:150] for c in classifications)
    explanations = ' | '.join(c.get('explanation', '')[:200] for c in classifications)
    pr_short     = '/'.join(url.split('/')[-3:])

    print(f'  [{dataset}] {pr_short} → {main_cats}')

    fix = {
        'PR_Link':       url,
        'Main_Category': main_cats,
        'Subcategory':   subcats,
        'Evidence':      evidences,
        'Explanation':   explanations,
    }

    if dataset == 'acc':
        fixed_acc.append(fix)
    else:
        fixed_rej.append(fix)

    time.sleep(0.3)

# Apply fixes to results
for fix in fixed_acc:
    acc_new_results.loc[acc_new_results['PR_Link'] == fix['PR_Link'], 'Main_Category'] = fix['Main_Category']
    acc_new_results.loc[acc_new_results['PR_Link'] == fix['PR_Link'], 'Subcategory']   = fix['Subcategory']
    acc_new_results.loc[acc_new_results['PR_Link'] == fix['PR_Link'], 'Evidence']      = fix['Evidence']
    acc_new_results.loc[acc_new_results['PR_Link'] == fix['PR_Link'], 'Explanation']   = fix['Explanation']

for fix in fixed_rej:
    rej_new_results.loc[rej_new_results['PR_Link'] == fix['PR_Link'], 'Main_Category'] = fix['Main_Category']
    rej_new_results.loc[rej_new_results['PR_Link'] == fix['PR_Link'], 'Subcategory']   = fix['Subcategory']
    rej_new_results.loc[rej_new_results['PR_Link'] == fix['PR_Link'], 'Evidence']      = fix['Evidence']
    rej_new_results.loc[rej_new_results['PR_Link'] == fix['PR_Link'], 'Explanation']   = fix['Explanation']

# Save updated checkpoints
acc_new_results.to_csv(CHECKPOINT_ACC, index=False)
rej_new_results.to_csv(CHECKPOINT_REJ, index=False)

# Check remaining errors
remaining_acc = acc_new_results[acc_new_results['Main_Category'] == 'ERROR']
remaining_rej = rej_new_results[rej_new_results['Main_Category'] == 'ERROR']
print(f'\nRemaining ERRORs — accepted: {len(remaining_acc)} | rejected: {len(remaining_rej)}')

  [acc] capture-app/pull/4109 → QUALITY ; REFACTORING ; INTEGRATION ; QUALITY ; INTEGRATION ; OBJECTIVE
  [acc] capture-app/pull/4112 → QUALITY ; REFACTORING ; REFACTORING ; OBJECTIVE ; INTEGRATION
  [acc] capture-app/pull/4244 → QUALITY ; QUALITY ; QUALITY ; QUALITY ; REFACTORING ; REFACTORING ; REFACTORING
  [acc] flux/pull/204 → QUALITY ; QUALITY ; REFACTORING ; REFACTORING ; REFACTORING ; REFACTORING ; QUALITY
  [acc] language/pull/13 → REFACTORING ; REFACTORING ; TESTING ; INTEGRATION ; INTEGRATION ; QUALITY ; REFACTORING
  [acc] mcp-config-converter/pull/6 → QUALITY ; QUALITY ; QUALITY ; QUALITY ; REFACTORING ; OBJECTIVE
  [acc] neuro-san-ui/pull/107 → QUALITY ; QUALITY ; QUALITY ; QUALITY ; INTEGRATION ; QUALITY
  [acc] airbyte/pull/72395 → QUALITY ; OBJECTIVE ; REFACTORING ; QUALITY ; REFACTORING ; OBJECTIVE ; REFACTORING ; TESTING ; QUALITY
  [acc] keboola-as-code/pull/2529 → TESTING ; QUALITY
  [acc] keboola-as-code/pull/2530 → QUALITY ; QUALITY ; QUALITY ; REFACTORING ; TEST

In [45]:
# Load current classified results (485 acc + 79 rej with independent reviews)
acc_classified = pd.read_csv(CHECKPOINT_ACC)
rej_classified = pd.read_csv(CHECKPOINT_REJ)

# Auto-tag PRs with no independent reviewer comments
def build_final(df_all, df_classified, label):
    classified_urls = set(df_classified['PR_Link'])
    auto_rows = df_all[~df_all['html_url'].isin(classified_urls)].copy()

    auto_list = []
    for _, row in auto_rows.iterrows():
        pr_short = '/'.join(row['html_url'].split('/')[-3:])
        auto_list.append({
            'PR':            pr_short,
            'PR_Link':       row['html_url'],
            'Main_Category': 'MANAGEMENT',
            'Subcategory':   'No review activity',
            'Evidence':      'No independent reviewer comments found.',
            'Explanation':   'All comments were from the PR author or no human comments exist.',
            'Label':         label,
            'Status':        'Accepted' if label == 'accepted' else 'Rejected'
        })

    final = pd.concat([df_classified, pd.DataFrame(auto_list)], ignore_index=True)
    final['#'] = range(1, len(final) + 1)
    return final

acc_final = build_final(acc, acc_classified, 'accepted')
rej_final = build_final(rej, rej_classified, 'rejected')

print(f'Final accepted : {len(acc_final)} (should be 842)')
print(f'Final rejected : {len(rej_final)} (should be 95)')

acc_final.to_csv('taxonomy_accepted_final.csv', index=False)
rej_final.to_csv('taxonomy_rejected_final.csv', index=False)

print('\nSaved: taxonomy_accepted_final.csv')
print('Saved: taxonomy_rejected_final.csv')

from google.colab import files
files.download('taxonomy_accepted_final.csv')
files.download('taxonomy_rejected_final.csv')


Final accepted : 842 (should be 842)
Final rejected : 95 (should be 95)

Saved: taxonomy_accepted_final.csv
Saved: taxonomy_rejected_final.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [46]:
import pandas as pd

# Load full AIDev dataset
df = pd.read_parquet("hf://datasets/hao-li/AIDev/all_pull_request.parquet")

# Filter refactoring PRs
refactor_keywords = r'refactor|refactoring|refactored'
closed_df = df[df['state'] == 'closed']
mask = (
    closed_df['title'].str.contains(refactor_keywords, case=False, na=False) |
    closed_df['body'].str.contains(refactor_keywords, case=False, na=False)
)
refactor_df = closed_df[mask]

# Add our new PRs
new_acc = pd.read_csv('accepted_with_reviews.csv')
new_rej = pd.read_csv('rejected_with_reviews.csv')
all_new = pd.concat([new_acc, new_rej])

# Combine and deduplicate repos
all_repos = pd.concat([
    refactor_df[['repo_url']],
    all_new[['repo_url']]
]).drop_duplicates()

print(f'Total refactoring PRs : {len(refactor_df) + len(all_new):,}')
print(f'Total unique repos    : {all_repos["repo_url"].nunique():,}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Total refactoring PRs : 33,762
Total unique repos    : 12,568


In [47]:
import pandas as pd

# Load full AIDev dataset
df = pd.read_parquet("hf://datasets/hao-li/AIDev/all_pull_request.parquet")

# Filter refactoring PRs
refactor_keywords = r'refactor|refactoring|refactored'
closed_df = df[df['state'] == 'closed']
mask = (
    closed_df['title'].str.contains(refactor_keywords, case=False, na=False) |
    closed_df['body'].str.contains(refactor_keywords, case=False, na=False)
)
refactor_df = closed_df[mask]

# Add our new PRs and deduplicate by id
new_acc = pd.read_csv('accepted_with_reviews.csv')
new_rej = pd.read_csv('rejected_with_reviews.csv')
all_new = pd.concat([new_acc, new_rej])

combined = pd.concat([refactor_df, all_new]).drop_duplicates(subset='id')

print(f'Total refactoring PRs : {len(combined):,}')
print(f'Total accepted PRs    : {combined[combined["merged_at"].notna()]["id"].nunique():,}')
print(f'Total rejected PRs    : {combined[combined["merged_at"].isna()]["id"].nunique():,}')
print(f'Total unique repos    : {combined["repo_url"].nunique():,}')
print(f'Accepted unique repos : {combined[combined["merged_at"].notna()]["repo_url"].nunique():,}')
print(f'Rejected unique repos : {combined[combined["merged_at"].isna()]["repo_url"].nunique():,}')

Total refactoring PRs : 33,316
Total accepted PRs    : 29,645
Total rejected PRs    : 3,671
Total unique repos    : 12,568
Accepted unique repos : 11,416
Rejected unique repos : 2,273
